___

# TEXT MINING - #5 TOPIC MODELLING
___

# Topic Model Evaluation & Comparison

In this notebook we will:
1. **Select the optimal number of topics (K)** using coherence scores
2. **Compare LDA vs NMF** on the same dataset
3. **Evaluate topic quality** using multiple metrics
4. **Visualize topics** with word clouds and other techniques

## Setup & Data Loading

In [ ]:
# Install required packages if needed
# !pip install gensim pyLDAvis wordcloud

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF

# gensim for coherence
import gensim
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

# visualization
from wordcloud import WordCloud

print("All packages loaded successfully!")

In [ ]:
# Load the NPR dataset
url = 'https://raw.githubusercontent.com/nluninja/text-mining-dataviz/refs/heads/main/4.%20Topic%20Modeling/npr.csv'
npr = pd.read_csv(url)
print(f"Loaded {len(npr)} documents")
npr.head()

## Preprocessing

We'll create both Count and TF-IDF representations:
- **CountVectorizer** for LDA (raw word counts)
- **TfidfVectorizer** for NMF (TF-IDF weighted)

In [ ]:
# For LDA: use CountVectorizer (raw counts)
count_vectorizer = CountVectorizer(
    max_df=0.90,      # Ignore words in >90% of docs
    min_df=5,         # Ignore words in <5 docs
    stop_words='english'
)
dtm_count = count_vectorizer.fit_transform(npr['Article'])

# For NMF: use TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.90,
    min_df=5,
    stop_words='english'
)
dtm_tfidf = tfidf_vectorizer.fit_transform(npr['Article'])

print(f"Count DTM shape: {dtm_count.shape}")
print(f"TF-IDF DTM shape: {dtm_tfidf.shape}")
print(f"Vocabulary size: {len(count_vectorizer.get_feature_names_out())}")

In [ ]:
# Prepare tokenized texts for gensim coherence calculation
# We need to tokenize using the same vocabulary as our vectorizer

def tokenize_for_coherence(documents, vectorizer):
    """Tokenize documents using the vectorizer's analyzer."""
    analyzer = vectorizer.build_analyzer()
    vocab = set(vectorizer.get_feature_names_out())
    tokenized = []
    for doc in documents:
        tokens = [t for t in analyzer(doc) if t in vocab]
        tokenized.append(tokens)
    return tokenized

tokenized_docs = tokenize_for_coherence(npr['Article'], count_vectorizer)
print(f"Example tokenized doc (first 20 tokens): {tokenized_docs[0][:20]}")

In [ ]:
# Create gensim dictionary for coherence calculation
gensim_dictionary = Dictionary(tokenized_docs)
print(f"Gensim dictionary size: {len(gensim_dictionary)}")

---
## 1. Model Selection: Finding the Optimal K

We'll train models with different numbers of topics and evaluate using:
- **Coherence Score (C_v)**: Measures semantic coherence of topics (higher = better)
- **Perplexity** (LDA only): Measures prediction ability (lower = better, but doesn't guarantee interpretability)

In [ ]:
def get_topic_words(model, vectorizer, n_words=10):
    """Extract top words for each topic."""
    feature_names = vectorizer.get_feature_names_out()
    topics = []
    for topic_weights in model.components_:
        top_word_indices = topic_weights.argsort()[:-n_words-1:-1]
        top_words = [feature_names[i] for i in top_word_indices]
        topics.append(top_words)
    return topics

def calculate_coherence(topic_words, tokenized_docs, dictionary, coherence='c_v'):
    """Calculate coherence score for a list of topics."""
    cm = CoherenceModel(
        topics=topic_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence=coherence
    )
    return cm.get_coherence()

In [ ]:
# Test different values of K
k_values = range(3, 16)

lda_coherence_scores = []
lda_perplexity_scores = []
nmf_coherence_scores = []
nmf_reconstruction_errors = []

print("Training models for different K values...")
print("This may take a few minutes...\n")

for k in k_values:
    print(f"K = {k}...", end=" ")
    
    # Train LDA
    lda = LatentDirichletAllocation(
        n_components=k,
        random_state=42,
        max_iter=10,
        learning_method='online'
    )
    lda.fit(dtm_count)
    
    # Train NMF
    nmf = NMF(
        n_components=k,
        random_state=42,
        max_iter=200,
        init='nndsvd'
    )
    nmf.fit(dtm_tfidf)
    
    # Get topic words
    lda_topics = get_topic_words(lda, count_vectorizer)
    nmf_topics = get_topic_words(nmf, tfidf_vectorizer)
    
    # Calculate coherence
    lda_coh = calculate_coherence(lda_topics, tokenized_docs, gensim_dictionary)
    nmf_coh = calculate_coherence(nmf_topics, tokenized_docs, gensim_dictionary)
    
    lda_coherence_scores.append(lda_coh)
    nmf_coherence_scores.append(nmf_coh)
    
    # LDA perplexity
    lda_perplexity_scores.append(lda.perplexity(dtm_count))
    
    # NMF reconstruction error
    nmf_reconstruction_errors.append(nmf.reconstruction_err_)
    
    print(f"LDA coherence: {lda_coh:.4f}, NMF coherence: {nmf_coh:.4f}")

print("\nDone!")

In [ ]:
# Plot coherence scores
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Coherence comparison
ax1 = axes[0, 0]
ax1.plot(k_values, lda_coherence_scores, 'b-o', label='LDA', linewidth=2, markersize=8)
ax1.plot(k_values, nmf_coherence_scores, 'r-s', label='NMF', linewidth=2, markersize=8)
ax1.set_xlabel('Number of Topics (K)', fontsize=12)
ax1.set_ylabel('Coherence Score (C_v)', fontsize=12)
ax1.set_title('Topic Coherence: LDA vs NMF', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Mark best K for each
best_lda_k = k_values[np.argmax(lda_coherence_scores)]
best_nmf_k = k_values[np.argmax(nmf_coherence_scores)]
ax1.axvline(x=best_lda_k, color='blue', linestyle='--', alpha=0.5, label=f'Best LDA K={best_lda_k}')
ax1.axvline(x=best_nmf_k, color='red', linestyle='--', alpha=0.5, label=f'Best NMF K={best_nmf_k}')

# LDA Perplexity
ax2 = axes[0, 1]
ax2.plot(k_values, lda_perplexity_scores, 'b-o', linewidth=2, markersize=8)
ax2.set_xlabel('Number of Topics (K)', fontsize=12)
ax2.set_ylabel('Perplexity (lower is better)', fontsize=12)
ax2.set_title('LDA Perplexity', fontsize=14)
ax2.grid(True, alpha=0.3)

# NMF Reconstruction Error
ax3 = axes[1, 0]
ax3.plot(k_values, nmf_reconstruction_errors, 'r-s', linewidth=2, markersize=8)
ax3.set_xlabel('Number of Topics (K)', fontsize=12)
ax3.set_ylabel('Reconstruction Error (lower is better)', fontsize=12)
ax3.set_title('NMF Reconstruction Error', fontsize=14)
ax3.grid(True, alpha=0.3)

# Summary table
ax4 = axes[1, 1]
ax4.axis('off')
summary_text = f"""
MODEL SELECTION SUMMARY
{'='*40}

LDA:
  Best K by coherence: {best_lda_k}
  Max coherence: {max(lda_coherence_scores):.4f}
  Perplexity at best K: {lda_perplexity_scores[best_lda_k - k_values[0]]:.2f}

NMF:
  Best K by coherence: {best_nmf_k}
  Max coherence: {max(nmf_coherence_scores):.4f}
  Reconstruction error at best K: {nmf_reconstruction_errors[best_nmf_k - k_values[0]]:.2f}

{'='*40}
NOTE: Higher coherence = more interpretable topics
      Perplexity doesn't always correlate with
      topic interpretability!
"""
ax4.text(0.1, 0.5, summary_text, fontsize=12, family='monospace',
         verticalalignment='center', transform=ax4.transAxes,
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('topic_model_selection.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nRecommendation: Use K={best_lda_k} for LDA and K={best_nmf_k} for NMF")

---
## 2. Train Final Models with Optimal K

In [ ]:
# Use a reasonable K based on coherence (you can adjust based on above results)
OPTIMAL_K = 7  # Adjust based on the coherence plot above

print(f"Training final models with K = {OPTIMAL_K}...")

# Final LDA model
lda_final = LatentDirichletAllocation(
    n_components=OPTIMAL_K,
    random_state=42,
    max_iter=20,
    learning_method='online',
    doc_topic_prior=0.1,    # alpha
    topic_word_prior=0.01   # beta
)
lda_doc_topics = lda_final.fit_transform(dtm_count)

# Final NMF model
nmf_final = NMF(
    n_components=OPTIMAL_K,
    random_state=42,
    max_iter=300,
    init='nndsvd'
)
nmf_doc_topics = nmf_final.fit_transform(dtm_tfidf)

print("Models trained!")

---
## 3. Evaluation Metrics

### 3.1 Topic Coherence

In [ ]:
# Get topic words for final models
lda_topic_words = get_topic_words(lda_final, count_vectorizer, n_words=15)
nmf_topic_words = get_topic_words(nmf_final, tfidf_vectorizer, n_words=15)

# Calculate different coherence metrics
coherence_metrics = ['c_v', 'c_npmi', 'u_mass']

print("Coherence Scores for Final Models")
print("=" * 50)

for metric in coherence_metrics:
    lda_coh = calculate_coherence(lda_topic_words, tokenized_docs, gensim_dictionary, coherence=metric)
    nmf_coh = calculate_coherence(nmf_topic_words, tokenized_docs, gensim_dictionary, coherence=metric)
    print(f"{metric.upper():8} | LDA: {lda_coh:7.4f} | NMF: {nmf_coh:7.4f}")

print("=" * 50)

### 3.2 Topic Diversity

Measures how different topics are from each other. Score of 1.0 means no word overlap.

In [ ]:
def calculate_topic_diversity(topic_words):
    """
    Calculate topic diversity: ratio of unique words to total words.
    Score of 1.0 = no word overlap between topics.
    """
    all_words = []
    for topic in topic_words:
        all_words.extend(topic)
    unique_words = len(set(all_words))
    total_words = len(all_words)
    return unique_words / total_words

lda_diversity = calculate_topic_diversity(lda_topic_words)
nmf_diversity = calculate_topic_diversity(nmf_topic_words)

print(f"Topic Diversity (higher = less overlap)")
print(f"  LDA: {lda_diversity:.4f}")
print(f"  NMF: {nmf_diversity:.4f}")

### 3.3 Per-Topic Coherence

Not all topics are equally coherent. Let's see which topics are best/worst.

In [ ]:
def calculate_per_topic_coherence(topic_words, tokenized_docs, dictionary):
    """Calculate coherence for each topic individually."""
    per_topic_scores = []
    for topic in topic_words:
        cm = CoherenceModel(
            topics=[topic],
            texts=tokenized_docs,
            dictionary=dictionary,
            coherence='c_v'
        )
        per_topic_scores.append(cm.get_coherence())
    return per_topic_scores

lda_per_topic = calculate_per_topic_coherence(lda_topic_words, tokenized_docs, gensim_dictionary)
nmf_per_topic = calculate_per_topic_coherence(nmf_topic_words, tokenized_docs, gensim_dictionary)

# Plot per-topic coherence
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(OPTIMAL_K)
width = 0.35

bars1 = ax.bar(x - width/2, lda_per_topic, width, label='LDA', color='steelblue')
bars2 = ax.bar(x + width/2, nmf_per_topic, width, label='NMF', color='coral')

ax.set_xlabel('Topic Number', fontsize=12)
ax.set_ylabel('Coherence Score (C_v)', fontsize=12)
ax.set_title('Per-Topic Coherence: LDA vs NMF', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([f'Topic {i}' for i in range(OPTIMAL_K)])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

---
## 4. Topic Visualization

### 4.1 Display Topics Side by Side

In [ ]:
def display_topics(topic_words, model_name):
    """Display topics in a formatted way."""
    print(f"\n{'='*60}")
    print(f"{model_name} TOPICS")
    print(f"{'='*60}")
    for idx, words in enumerate(topic_words):
        print(f"\nTopic {idx}: {', '.join(words[:10])}")

display_topics(lda_topic_words, "LDA")
display_topics(nmf_topic_words, "NMF")

### 4.2 Word Clouds for Each Topic

In [ ]:
def plot_topic_wordclouds(model, vectorizer, model_name, n_topics):
    """Generate word clouds for each topic."""
    feature_names = vectorizer.get_feature_names_out()
    
    # Calculate grid size
    n_cols = 3
    n_rows = (n_topics + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axes = axes.flatten()
    
    for topic_idx, topic in enumerate(model.components_):
        # Create word frequency dict
        word_freq = dict(zip(feature_names, topic))
        
        # Generate word cloud
        wc = WordCloud(
            width=400, height=300,
            background_color='white',
            colormap='viridis',
            max_words=50
        ).generate_from_frequencies(word_freq)
        
        axes[topic_idx].imshow(wc, interpolation='bilinear')
        axes[topic_idx].set_title(f'Topic {topic_idx}', fontsize=14)
        axes[topic_idx].axis('off')
    
    # Hide empty subplots
    for idx in range(n_topics, len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle(f'{model_name} Topic Word Clouds', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.savefig(f'{model_name.lower()}_wordclouds.png', dpi=150, bbox_inches='tight')
    plt.show()

# Plot word clouds for both models
plot_topic_wordclouds(lda_final, count_vectorizer, "LDA", OPTIMAL_K)
plot_topic_wordclouds(nmf_final, tfidf_vectorizer, "NMF", OPTIMAL_K)

### 4.3 Topic Distribution in Corpus

In [ ]:
# Get dominant topic for each document
lda_dominant = lda_doc_topics.argmax(axis=1)
nmf_dominant = nmf_doc_topics.argmax(axis=1)

# Count documents per topic
lda_counts = pd.Series(lda_dominant).value_counts().sort_index()
nmf_counts = pd.Series(nmf_dominant).value_counts().sort_index()

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(lda_counts.index, lda_counts.values, color='steelblue')
axes[0].set_xlabel('Topic', fontsize=12)
axes[0].set_ylabel('Number of Documents', fontsize=12)
axes[0].set_title('LDA: Document Distribution Across Topics', fontsize=14)
axes[0].set_xticks(range(OPTIMAL_K))

axes[1].bar(nmf_counts.index, nmf_counts.values, color='coral')
axes[1].set_xlabel('Topic', fontsize=12)
axes[1].set_ylabel('Number of Documents', fontsize=12)
axes[1].set_title('NMF: Document Distribution Across Topics', fontsize=14)
axes[1].set_xticks(range(OPTIMAL_K))

plt.tight_layout()
plt.show()

print("\nDocument counts per topic:")
print(f"{'Topic':<10} {'LDA':<10} {'NMF':<10}")
print("-" * 30)
for i in range(OPTIMAL_K):
    lda_c = lda_counts.get(i, 0)
    nmf_c = nmf_counts.get(i, 0)
    print(f"{i:<10} {lda_c:<10} {nmf_c:<10}")

### 4.4 Representative Documents for Each Topic

In [ ]:
def show_representative_docs(doc_topics, documents, topic_words, n_docs=2, model_name="Model"):
    """Show most representative documents for each topic."""
    print(f"\n{'='*80}")
    print(f"{model_name}: Representative Documents for Each Topic")
    print(f"{'='*80}")
    
    for topic_idx in range(doc_topics.shape[1]):
        # Get documents most associated with this topic
        top_doc_indices = doc_topics[:, topic_idx].argsort()[-n_docs:][::-1]
        
        print(f"\n--- Topic {topic_idx}: {', '.join(topic_words[topic_idx][:5])} ---")
        
        for rank, doc_idx in enumerate(top_doc_indices):
            score = doc_topics[doc_idx, topic_idx]
            doc_text = documents.iloc[doc_idx][:300] + "..."
            print(f"\n  Doc {rank+1} (score: {score:.3f}):")
            print(f"  {doc_text}")

show_representative_docs(lda_doc_topics, npr['Article'], lda_topic_words, n_docs=1, model_name="LDA")

---
## 5. pyLDAvis Interactive Visualization (LDA only)

In [ ]:
import pyLDAvis
import pyLDAvis.lda_model

pyLDAvis.enable_notebook()

# Create visualization
vis = pyLDAvis.lda_model.prepare(lda_final, dtm_count, count_vectorizer)
vis

---
## 6. Summary Comparison

In [ ]:
# Final summary
lda_final_coherence = calculate_coherence(lda_topic_words, tokenized_docs, gensim_dictionary, 'c_v')
nmf_final_coherence = calculate_coherence(nmf_topic_words, tokenized_docs, gensim_dictionary, 'c_v')

print("\n" + "="*60)
print("FINAL MODEL COMPARISON SUMMARY")
print("="*60)
print(f"\nNumber of topics (K): {OPTIMAL_K}")
print(f"Number of documents: {len(npr)}")
print(f"Vocabulary size: {len(count_vectorizer.get_feature_names_out())}")
print("\n" + "-"*60)
print(f"{'Metric':<30} {'LDA':<15} {'NMF':<15}")
print("-"*60)
print(f"{'Coherence (C_v)':<30} {lda_final_coherence:<15.4f} {nmf_final_coherence:<15.4f}")
print(f"{'Topic Diversity':<30} {lda_diversity:<15.4f} {nmf_diversity:<15.4f}")
print(f"{'Perplexity':<30} {lda_final.perplexity(dtm_count):<15.2f} {'N/A':<15}")
print(f"{'Reconstruction Error':<30} {'N/A':<15} {nmf_final.reconstruction_err_:<15.2f}")
print("-"*60)

# Recommendation
if lda_final_coherence > nmf_final_coherence:
    winner = "LDA"
    margin = lda_final_coherence - nmf_final_coherence
else:
    winner = "NMF"
    margin = nmf_final_coherence - lda_final_coherence

print(f"\nBased on coherence: {winner} performs better by {margin:.4f}")
print("\nNote: Always inspect topics qualitatively - metrics aren't everything!")
print("="*60)

---
## Key Takeaways

1. **Model Selection**: Use coherence scores (C_v) to find optimal K - look for the peak or elbow

2. **Perplexity vs Coherence**: Lower perplexity doesn't guarantee better topics! Coherence correlates better with human judgment

3. **LDA vs NMF**: 
   - LDA: Probabilistic, uses raw counts, good theoretical foundation
   - NMF: Faster, deterministic with NNDSVD, uses TF-IDF

4. **Multiple Metrics**: Use coherence, diversity, AND qualitative inspection

5. **Representative Documents**: Always check if top documents for each topic make sense